<a href="https://colab.research.google.com/github/Omar-Qaid/Python/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import Dataset
import pandas as pd
import requests
from io import BytesIO
from huggingface_hub import notebook_login

# OPTION 1: Run this once to login interactively
# notebook_login()

# OPTION 2: Or paste your token here:
HF_TOKEN = ""

# Column names
my_column_names = [
    "RequestId", "ProcessName",  "EmployeeName",
    "EmployeeCode", "CreatedDate", "FinishedDate",
    "ControlLabel", "ControlValue", "ActualWeight"
]

# Load using requests with the authorization header
url = "https://huggingface.co/datasets/almojahid/dkoon/resolve/main/fineTuing.csv"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    # Load into pandas
    df = pd.read_csv(
        BytesIO(response.content),
        sep=";",
        names=my_column_names,
        encoding="windows-1256",
        header=None,
        on_bad_lines='skip',
        low_memory=False
    )

    # Convert ActualWeight to numeric
    df['ActualWeight'] = pd.to_numeric(df['ActualWeight'], errors='coerce').fillna(0.0)

    # Convert all other columns to string to avoid ArrowTypeError with mixed types
    for col in df.columns:
        if col != 'ActualWeight':
            df[col] = df[col].astype(str)

    # Convert to HuggingFace dataset
    dataset = Dataset.from_pandas(df)

    # Define your prompt template
    process_prompt = """### Process: {} ({})

Employee: {} ({})
Request ID: {}
Status: Started on {} and finished on {}
Detail: {} - {}

### Summary:
The employee {} handled the "{}" request.
The control value recorded was {} with an actual weight of {}.
"""

    EOS_TOKEN = tokenizer.eos_token

    # Mapping function
    def formatting_prompts_func(examples):
        texts = []
        for i in range(len(examples["RequestId"])):
            text = process_prompt.format(
                str(examples["ProcessName"][i]), str(examples["ProcessName"][i]),
                str(examples["EmployeeName"][i]), str(examples["EmployeeCode"][i]),
                str(examples["RequestId"][i]),
                str(examples["CreatedDate"][i]), str(examples["FinishedDate"][i]),
                str(examples["ControlLabel"][i]), str(examples["ControlValue"][i]),
                str(examples["EmployeeName"][i]), str(examples["ProcessName"][i]),
                str(examples["ControlValue"][i]), str(examples["ActualWeight"][i])
            ) + EOS_TOKEN
            texts.append(text)
        return {"text": texts}

    # Apply formatting
    dataset = dataset.map(formatting_prompts_func, batched=True)
    print("Dataset loaded and formatted successfully.")
else:
    print(f"Failed to download file. Status code: {response.status_code}. Please ensure you have replaced 'YOUR_HF_TOKEN_HERE' with a valid token.")

In [ ]:
dataset['text']

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
try:
    trainer_stats = trainer.train()
    print("Training completed successfully!")
    print(f"Training stats: {trainer_stats}")
except RuntimeError as e:
    print(f"Runtime error occurred: {e}")

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")

In [ ]:
model.push_to_hub_gguf("almojahid/dkhoun", tokenizer, quantization_method = "f16", token = "")